In [6]:
%matplotlib inline
import os
import sys
import glob
import numpy as np
import pandas as pd
from datetime import datetime
import xarray as xr
from multiprocessing import Pool

from read_deterministic_data import load_GFS_datasets, load_ECMWF_datasets, read_preprocessed_IVT_data
from calc_funcs import format_timedelta_to_HHMMSS
from cw3e_tools import remove_tmp_data_files
from plotter import plot_time_height_meteograms

model_name = 'GFS' ## GFS or ECMWF
print('Creating WV Flux Meteograms for {0}'.format(model_name))

## FIRST LOOP - LEAD TIME ##
if model_name == 'ECMWF':
    arr1 = np.arange(0, 72+3, 3)
    arr2 = np.arange(78, 168+6, 6)
    F_lst = np.concatenate((arr1, arr2), axis=0)
elif model_name == 'GFS':
    F_lst = np.arange(0, 168+3, 3)

fdate='2025021012' ## set this to None to get most recently downloaded data

Creating WV Flux Meteograms for GFS


In [7]:
#################################
### CHECK TO REMOVE TMP FILES ###
#################################
print('Removing tmp intermediate data files...') 
# Specify the directory and the pattern
tmp_directory = "/data/projects/operations/wvflux_meteograms/data/tmp/"
pattern = "tmp_{0}*.nc".format(model_name)  # Delete all tmp files
remove_tmp_data_files(tmp_directory, pattern)

Removing tmp intermediate data files...


In [8]:
# ### 000 - prate
# ### 003 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp
# ### 006 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp

# filename = '/data/projects/external_datasets/GFS/processed/{0}/{0}_F009.grb2'.format(fdate)
# ds = xr.open_dataset(filename,
#                         engine='cfgrib',backend_kwargs={"indexpath": ''}, filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'})
# ds

In [9]:
%%time
####################################
### TEST WITH A FEW NON-PARALLEL ###
####################################
for i, F in enumerate(F_lst[0:3]):
    start_time = pd.Timestamp.today()
    #######################
    ### LOAD MODEL DATA ###
    #######################
    print('... Loading {0} data for {1} hour lead'.format(model_name, F))
    if model_name == 'ECMWF':
        s = load_ECMWF_datasets(F=F, fdate=fdate)
        model_data= s.calc_vars()

    elif model_name == 'GFS':
        s = load_GFS_datasets(F=F, fdate=fdate)
        model_data = s.calc_vars()
        
model_data

... Loading GFS data for 0 hour lead
Preprocessing 0 ...
... Loading GFS data for 3 hour lead
Preprocessing 3 ...
... Loading GFS data for 6 hour lead
Preprocessing 6 ...
CPU times: user 29.3 s, sys: 2.99 s, total: 32.3 s
Wall time: 1min 4s


<xarray.Dataset>
Dimensions:                (isobaricInhPa: 33, latitude: 44, longitude: 64)
Coordinates:
    time                   datetime64[ns] 2025-02-10T12:00:00
    step                   timedelta64[ns] 06:00:00
  * isobaricInhPa          (isobaricInhPa) float64 1e+03 975.0 ... 225.0 200.0
  * latitude               (latitude) float64 26.0 27.0 28.0 ... 67.0 68.0 69.0
  * longitude              (longitude) float64 186.0 187.0 188.0 ... 248.0 249.0
    valid_time             datetime64[ns] 2025-02-10T18:00:00
    atmosphereSingleLayer  float64 ...
    surface                float64 0.0
    isothermZero           float64 ...
Data variables:
    u                      (isobaricInhPa, latitude, longitude) float32 9.014...
    v                      (isobaricInhPa, latitude, longitude) float32 11.12...
    r                      (isobaricInhPa, latitude, longitude) float32 84.1 ...
    pwat                   (latitude, longitude) float32 ...
    t                      (isobaricInhPa, latitude, longitude) float32 296.3...
    orog                   (latitude, longitude) float32 ...
    sp                     (latitude, longitude) float32 ...
    gh                     (latitude, longitude) float32 4.387e+03 ... 0.0
    tp                     (latitude, longitude) float32 ...
    wvflux                 (isobaricInhPa, latitude, longitude) float64 0.253...
    freezing_level_pres    (latitude, longitude) float32 585.8 ... 1.013e+03
Attributes: (12/29)
    GRIB_paramId:                             131
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                U component of wind
    GRIB_shortName:                           u
    GRIB_units:                               m s**-1
    long_name:                                U component of wind
    units:                                    m s**-1
    standard_name:                            eastward_wind

In [5]:
############################################
### PREPROCESS INTERMEDIATE GFS OR ECMWF ###
############################################
print('...preprocess intermediate data ...')

def multiP_preprocess_intermediate(model_name, F, fdate):
    ##############################
    ### PREP INTERMEDIATE DATA ###
    ##############################
    print('... Loading data for {0} hour lead'.format(F))
    if model_name == 'ECMWF':
        s = load_ECMWF_datasets(F=F, fdate=fdate)
        model_data = s.calc_vars()

    elif model_name == 'GFS':
        s = load_GFS_datasets(F=F, fdate=fdate)
        model_data, tmp = s.calc_vars()

if __name__ == '__main__':
    with Pool(processes=30) as pool:
        for F in F_lst:
            result = pool.apply_async(func=multiP_preprocess_intermediate,args=(model_name, F, fdate))
        pool.close()
        pool.join()

...preprocess intermediate data ...
... Loading data for 0 hour lead... Loading data for 6 hour lead... Loading data for 9 hour lead... Loading data for 3 hour lead... Loading data for 12 hour lead... Loading data for 15 hour lead... Loading data for 18 hour lead... Loading data for 27 hour lead... Loading data for 24 hour lead... Loading data for 21 hour lead... Loading data for 33 hour lead... Loading data for 30 hour lead... Loading data for 36 hour lead
... Loading data for 42 hour lead... Loading data for 45 hour lead
... Loading data for 48 hour lead... Loading data for 51 hour lead

... Loading data for 54 hour lead... Loading data for 39 hour lead

... Loading data for 57 hour lead
... Loading data for 63 hour lead
... Loading data for 60 hour lead
... Loading data for 66 hour lead
... Loading data for 72 hour lead
Preprocessing 0 ...
... Loading data for 87 hour lead... Loading data for 84 hour lead... Loading data for 75 hour leadPreprocessing 9 ...

... Loading data for 69 h

/home/dnash/miniconda3/envs/SEAK-impacts/lib/python3.12/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
Struct() takes at most 1 argument (3 given)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)
/home/dnash/miniconda3/envs/SEAK-impacts/lib/python3.12/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
Struct() takes at most 1 argument (3 given)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)
/home/dnash/miniconda3/envs/SEAK-impacts/lib/python3.12/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
Struct() takes at most 1 argument (3 given)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)
/home/dnash/miniconda3/envs/SEAK-impacts/lib/python3.12/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gini' loading failed:
Struct() takes at most 1 argument (3 given)
  warnings.war

... Loading data for 90 hour lead
Preprocessing 90 ...
... Loading data for 93 hour lead
Preprocessing 93 ...
... Loading data for 96 hour lead
Preprocessing 96 ...
... Loading data for 99 hour lead
Preprocessing 99 ...
... Loading data for 102 hour lead
Preprocessing 102 ...
... Loading data for 105 hour lead
Preprocessing 105 ...
... Loading data for 108 hour lead
Preprocessing 108 ...
... Loading data for 111 hour lead
Preprocessing 111 ...
... Loading data for 114 hour lead
Preprocessing 114 ...
... Loading data for 117 hour lead
Preprocessing 117 ...
... Loading data for 120 hour lead
Preprocessing 120 ...... Loading data for 123 hour lead

Preprocessing 123 ...
... Loading data for 129 hour lead... Loading data for 126 hour lead

Preprocessing 129 ...Preprocessing 126 ...

... Loading data for 132 hour lead
Preprocessing 132 ...
... Loading data for 135 hour lead
Preprocessing 135 ...
... Loading data for 138 hour lead
Preprocessing 138 ...
... Loading data for 141 hour lead
Prep

In [34]:
##############################
### LOAD INTERMEDIATE DATA ###
##############################
tmp_directory = "/home/dnash/comet_data/tmp/"
files = glob.glob(tmp_directory + 'tmp_{0}_*.nc'.format(model_name))
sorted_files = sorted(files)
ds = xr.open_mfdataset(sorted_files, engine='netcdf4', concat_dim="step", combine='nested')

## need to fix precipitation
if model_name == 'ECMWF':
    tp = ds.tp.fillna(0) # make sure nan at beginning is 0 to do calculation right
    tp = tp.diff(dim='step') # calculate difference between time steps
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, tp]) # merge dataset with new tp
    
if model_name == 'GFS':
    ts_3hr = pd.timedelta_range(start='0 day', periods=57, freq='3H')
    ts_6hr = pd.timedelta_range(start='0 day', periods=29, freq='6H')
    tp = ds.tp ## pull out tp
    prec_3hr = tp.sel(step=ts_3hr[1::2]) ## grab only the 3hr values
    tp2 = tp.diff(dim='step') ## calculate difference in precip
    ## the values for 6hr timesteps are correct, the values for 3hr timesteps are incorrect
    prec_6hr = tp2.sel(step=ts_6hr[1:]) # grab only the 6hr values
    new_prec = prec_3hr.combine_first(prec_6hr) # combine the correct 3hr values with the correct 6hr values
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, new_prec]) # merge dataset with new tp
    
ivt = read_preprocessed_IVT_data(model=model_name, F_lst=F_lst, fdate=pd.to_datetime(ds.time.values).strftime('%Y%m%d%H'))
ds = ds.assign(ivt=(['step','latitude','longitude'],ivt.ivt.values))

<xarray.Dataset>
Dimensions:                (step: 57, isobaricInhPa: 33, latitude: 44,
                            longitude: 64)
Coordinates:
  * step                   (step) timedelta64[ns] 00:00:00 ... 7 days 00:00:00
  * isobaricInhPa          (isobaricInhPa) float64 1e+03 975.0 ... 225.0 200.0
  * latitude               (latitude) float64 26.0 27.0 28.0 ... 67.0 68.0 69.0
  * longitude              (longitude) float64 186.0 187.0 188.0 ... 248.0 249.0
    time                   datetime64[ns] 2025-02-07T12:00:00
    valid_time             (step) datetime64[ns] 2025-02-07T12:00:00 ... 2025...
    atmosphereSingleLayer  float64 0.0
    surface                float64 0.0
    isothermZero           float64 0.0
Data variables:
    u                      (step, isobaricInhPa, latitude, longitude) float32 dask.array<chunksize=(1, 33, 44, 64), meta=np.ndarray>
    v                      (step, isobaricInhPa, latitude, longitude) float32 dask.array<chunksize=(1, 33, 44, 64), meta=np.ndarray>
    r                      (step, isobaricInhPa, latitude, longitude) float32 dask.array<chunksize=(1, 33, 44, 64), meta=np.ndarray>
    pwat                   (step, latitude, longitude) float32 dask.array<chunksize=(1, 44, 64), meta=np.ndarray>
    t                      (step, isobaricInhPa, latitude, longitude) float32 dask.array<chunksize=(1, 33, 44, 64), meta=np.ndarray>
    orog                   (step, latitude, longitude) float32 dask.array<chunksize=(1, 44, 64), meta=np.ndarray>
    sp                     (step, latitude, longitude) float32 dask.array<chunksize=(1, 44, 64), meta=np.ndarray>
    gh                     (step, latitude, longitude) float32 dask.array<chunksize=(1, 44, 64), meta=np.ndarray>
    wvflux                 (step, isobaricInhPa, latitude, longitude) float64 dask.array<chunksize=(1, 33, 44, 64), meta=np.ndarray>
    freezing_level_pres    (step, latitude, longitude) float32 dask.array<chunksize=(1, 44, 64), meta=np.ndarray>
    tp                     (step, latitude, longitude) float32 dask.array<chunksize=(3, 44, 64), meta=np.ndarray>
    ivt                    (step, latitude, longitude) float64 402.4 ... 28.03
Attributes: (12/29)
    GRIB_paramId:                             131
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                U component of wind
    GRIB_shortName:                           u
    GRIB_units:                               m s**-1
    long_name:                                U component of wind
    units:                                    m s**-1
    standard_name:                            eastward_wind

In [37]:
plot_time_height_meteograms(ds, 'wvflux', 39, 360-120, model_name, duration=7)

In [3]:
import numpy as np
lat_lst = np.arange(26, 51, 1)
lon_lst = np.arange(111, 128, 1)
var_lst = ['r', 'wvflux']
dur_lst = [3, 7]

arglst_USWEST = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                arglst_USWEST.append((x,y, varname, dur))

lat_lst = np.arange(51, 70, 1)
lon_lst = np.arange(130, 175, 1)               
arglst_AK = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                arglst_AK.append((x,y, varname, dur))
                
final_arglst = arglst_USWEST + arglst_AK

for i, argval in enumerate(final_arglst[:2]):
    lon, lat, varname, dur = argval
    print(lon, lat, varname, dur)

111 26 r 3
111 26 r 7
